Wiley

In [48]:
import glob
import pandas as pd
# Pega todos os arquivos txt (ajuste o caminho se necessário)
arquivos = glob.glob("C:/Users/Hychiro/Documents/Mestrado/Mestrado/Mestrado/Review/SLR-BCI-temporal-filters/raw_data/wiley_export_issue_*_Mar_18-2026.txt")
artigos = []
for arquivo in arquivos:
    with open(arquivo, "r", encoding="utf-8") as f:
        dados = {}
        for linha in f:
            linha = linha.strip()
            if not linha:
                continue
            if " - " in linha:
                chave, valor = linha.split(" - ", 1)
                # Se já existe a chave, acumula em lista (ex: vários autores)
                if chave in dados:
                    if isinstance(dados[chave], list):
                        dados[chave].append(valor)
                    else:
                        dados[chave] = [dados[chave], valor]
                else:
                    dados[chave] = valor
            # ER marca o fim do artigo
            if linha.startswith("ER"):
                # Normaliza listas em strings separadas por ";"
                for k, v in dados.items():
                    if isinstance(v, list):
                        dados[k] = "; ".join(v)
                artigos.append(dados)
                dados = {}

dfWiley = pd.DataFrame(artigos)

# Remove espaços extras dos nomes das colunas
dfWiley.columns = dfWiley.columns.str.strip()

# Mantém apenas os campos desejados
dfWiley = dfWiley[["AU", "TI", "PY"]]

# Renomeia colunas
dfWiley = dfWiley.rename(columns={
    "AU": "Autores",
    "TI": "Titulo",
    "PY": "Ano de Publicacao"
})

dfWiley.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Autores            166 non-null    object
 1   Titulo             166 non-null    object
 2   Ano de Publicacao  166 non-null    object
dtypes: object(3)
memory usage: 4.0+ KB


IEEE

In [49]:
import pandas as pd

# Caminho do arquivo
caminho = r"C:/Users/Hychiro/Documents/Mestrado/Mestrado/Mestrado/Review/SLR-BCI-temporal-filters/raw_data/ieee_export_Mar_18-2026.csv"

# Lê o CSV
dfIEEE = pd.read_csv(caminho, encoding="utf-8")

# Mantém apenas as colunas desejadas
colunas_desejadas = ['Document Title', 'Authors', 'Publication Year', 'Abstract', 'Author Keywords']
dfIEEE_reduzido = dfIEEE.reindex(columns=colunas_desejadas)

# Junta os campos relevantes em um só texto
dfIEEE_reduzido['texto'] = (
    dfIEEE_reduzido['Document Title'].fillna('') + ' ' +
    dfIEEE_reduzido['Abstract'].fillna('') + ' ' +
    dfIEEE_reduzido['Author Keywords'].fillna('')
).str.lower()

# Define os grupos da busca (equivalente ao AND entre blocos)

cond1 = dfIEEE_reduzido['texto'].str.contains(
    r"electroencephalogram|electroencephalography|\beeg\b", regex=True
)

cond2 = dfIEEE_reduzido['texto'].str.contains(
    r"motor imagery|sensorimotor rhythm|motor image|imagined movement|\bmi\b|movement imagination",
    regex=True
)

cond3 = dfIEEE_reduzido['texto'].str.contains(
    r"brain computer interface|brain computer interfaces|\bbci\b|brain machine interface|brain machine interfaces|\bbmi\b",
    regex=True
)

cond4 = dfIEEE_reduzido['texto'].str.contains(
    r"temporal filter|time[- ]domain filter|time[- ]frequency|digital signal|adaptive filter|kernel filter|multikernel|multi[- ]kernel|nonlinear filter|linear filter|stochastic filter|bayesian filter|kalman filter|particle filter|sequential monte carlo|recursive least square|\brls\b|least mean square|\blms\b|finite impulse response|\bfir\b|infinite impulse response|\biir\b|filter bank|band[- ]pass filter|butterworth|notch filter|gabor filter|wavelet transform|short[- ]time fourier|\bstft\b|fast fourier|\bfft\b|empirical mode decomposition|\bemd\b|noise reduction|denoising|artifact removal|artefact removal",
    regex=True
)

cond5 = dfIEEE_reduzido['texto'].str.contains(
    r"pattern recognition|pattern classification|pattern detection|classification|online system|online processing|on-line system|real[- ]time system|real[- ]time processing|low[- ]latency|digital system|digital processing system",
    regex=True
)

# Aplica todos os ANDs
dfIEEE_filtrado = dfIEEE_reduzido[cond1 & cond2 & cond3 & cond4 & cond5]

# Resultado
print(f"Total de artigos filtrados: {len(dfIEEE_filtrado)}")
print(dfIEEE_filtrado.head())

#pega agora só os campos relevantes para análise
colunas_desejadas = ['Document Title', 'Authors', 'Publication Year']
dfIEEE_filtrado_reduzido = dfIEEE_filtrado.reindex(columns=colunas_desejadas)

# Renomeia para nomes mais claros
dfIEEE_filtrado_reduzido = dfIEEE_filtrado_reduzido.rename(columns={
    'Document Title': 'Titulo',
    'Authors': 'Autores',
    'Publication Year': 'Ano de Publicacao',
})

dfIEEE_filtrado_reduzido.info()

Total de artigos filtrados: 426
                                      Document Title  \
0  Efficient EEG-Based Motor Imagery Classificati...   
1  FA-EEGNet: Enhancing EEGNet with Frequency-Ada...   
5  Decoding of Intuitive Motor Imagery Paradigm B...   
8  Scale Specific Riemannian Features with Graph ...   
9  NeuroClean: A Benchmarking and Optimization Fr...   

                                     Authors  Publication Year  \
0        M. Nadeem; S. M. Qaisar; U. S. Khan              2026   
1       M. H. Mim; D. R. Pranto; M. M. Hasan              2026   
5               L. Pan; D. Liu; J. Li; J. Li              2026   
8  Balendra; A. Pathak; N. Sharma; S. Sharma              2025   
9                      P. Pawar; N. Kulkarni              2025   

                                            Abstract  \
0  This paper presents a Brain-computer interface...   
1  Brain-computer interfaces (BCIs) can enhance p...   
5  Motor imagery (MI) based brain-computer interf...   
8  Motor i

Pubmed

In [50]:
# Caminho do arquivo
caminho = r"C:/Users/Hychiro/Documents/Mestrado/Mestrado/Mestrado/Review/SLR-BCI-temporal-filters/raw_data/pubmed_export_Mar_18-2026.csv"

# Lê o CSV
dfPubmed = pd.read_csv(caminho, encoding="utf-8")

# Seleciona apenas as colunas desejadas
colunas_desejadas = ["Title", "Authors", "Publication Year"]
dfPubmed_reduzido = dfPubmed.reindex(columns=colunas_desejadas)

# Renomeia para nomes mais claros
dfPubmed_reduzido = dfPubmed_reduzido.rename(columns={
    "Title": "Titulo",
    "Authors": "Autores",
    "Publication Year": "Ano de Publicacao",
    
})

dfPubmed_reduzido.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Titulo             13 non-null     object
 1   Autores            13 non-null     object
 2   Ano de Publicacao  13 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 444.0+ bytes


Scopus

In [51]:
caminho = r"C:/Users/Hychiro/Documents/Mestrado/Mestrado/Mestrado/Review/SLR-BCI-temporal-filters/raw_data/scopus_export_Mar_18-2026.csv"

# Lê o CSV
dfScopus = pd.read_csv(caminho, encoding="utf-8")

# Seleciona apenas as colunas desejadas
colunas_desejadas = ["Title", "Authors", "Year"]
dfScopus_reduzido = dfScopus.reindex(columns=colunas_desejadas)

# Renomeia para nomes mais claros
dfScopus_reduzido = dfScopus_reduzido.rename(columns={
    "Title": "Titulo",
    "Authors": "Autores",
    "Year": "Ano de Publicacao",
    
})

dfScopus_reduzido.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Titulo             123 non-null    object
 1   Autores            122 non-null    object
 2   Ano de Publicacao  123 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 3.0+ KB


In [52]:
#adiciona uma coluna para identificar a fonte dos dados
dfWiley["Fonte"] = "Wiley"
dfIEEE_filtrado_reduzido["Fonte"] = "IEEE"
dfPubmed_reduzido["Fonte"] = "PubMed"
dfScopus_reduzido["Fonte"] = "Scopus"

In [53]:
# unir todos os dataframes em um só
df_completo = pd.concat([dfIEEE_filtrado_reduzido, dfPubmed_reduzido, dfScopus_reduzido], ignore_index=True)

df_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562 entries, 0 to 561
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Titulo             562 non-null    object
 1   Autores            561 non-null    object
 2   Ano de Publicacao  562 non-null    int64 
 3   Fonte              562 non-null    object
dtypes: int64(1), object(3)
memory usage: 17.7+ KB


In [54]:
df_completo.head()

,Titulo,Autores,Ano de Publicacao,Fonte
0,Efficient EEG-Based Motor Imagery Classificati...,M. Nadeem; S. M. Qaisar; U. S. Khan,2026,IEEE
1,FA-EEGNet: Enhancing EEGNet with Frequency-Ada...,M. H. Mim; D. R. Pranto; M. M. Hasan,2026,IEEE
2,Decoding of Intuitive Motor Imagery Paradigm B...,L. Pan; D. Liu; J. Li; J. Li,2026,IEEE
3,Scale Specific Riemannian Features with Graph ...,Balendra; A. Pathak; N. Sharma; S. Sharma,2025,IEEE
4,NeuroClean: A Benchmarking and Optimization Fr...,P. Pawar; N. Kulkarni,2025,IEEE


In [55]:
# passar df completo para csv em um local específico structured_data
caminho = r"C:/Users/Hychiro/Documents/Mestrado/Mestrado/Mestrado/Review/SLR-BCI-temporal-filters/structured_data/dataframe_completo.csv"
df_completo.to_csv(caminho, index=False, encoding="utf-8")    

In [56]:
# passar strip no df completo para remover espaços extras nos titulos
df_completo["Titulo"] = df_completo["Titulo"].str.strip() 

# vizualisar duplicatas de titulo no df completo. Mostrar todas as colunas para verificar se são realmente duplicatas ou apenas títulos iguais com autores ou anos diferentes
print("Títulos duplicados:")
df_duplicatas = df_completo[df_completo.duplicated(subset=["Titulo"], keep=False)].sort_values("Titulo")
df_duplicatas

Títulos duplicados:


,Titulo,Autores,Ano de Publicacao,Fonte
494,A Deep Learning Scheme for Motor Imagery Class...,Lu N.; Li T.; Ren X.; Miao H.,2017,Scopus
290,A Deep Learning Scheme for Motor Imagery Class...,N. Lu; T. Li; X. Ren; H. Miao,2017,IEEE
59,A Novel Classification Model Based on DWT and ...,Y. Yang; Z. Chen; W. Li; Y. Ma,2024,IEEE
479,A Novel Classification Model Based on DWT and ...,Yang Y.; Chen Z.; Li W.; Ma Y.,2024,Scopus
519,A comparative analysis of masking empirical mo...,Jaipriya D.; Sriharipriya K.C.,2022,Scopus
...,...,...,...,...
299,Wavelet based feature extraction for classific...,F. Sherwani; S. Shanta; B. S. K. K. Ibrahim; M...,2016,IEEE
547,[Research of classification about BCI based on...,Qiao J.; Hu P.; Hong J.,2014,Scopus
431,[Research of classification about BCI based on...,"Qiao J, Hu P, Hong J.",2014,PubMed
443,iTa-DFiE: An Innovative Tensor Algebra-Based D...,Nguyen N.A.T.; Tao Q.-B.; Yang H.-J.; Huynh H....,2024,Scopus


In [57]:
# remover primeira instancia de duplicata dando preferencia para a fonte Pubmed, depois Scopus, depois Wiley e por último Scopus
#ordenar o df de duplicatas pela coluna "Fonte" para garantir a ordem de preferência Pubmed > Scopus > Wiley > IEEE
df_duplicatas["Fonte"] = pd.Categorical(df_duplicatas["Fonte"], categories=["PubMed", "Scopus", "Wiley", "IEEE"], ordered=True)
df_duplicatas = df_duplicatas.sort_values("Fonte")
print(df_duplicatas.shape)
df_duplicatas.drop_duplicates(subset=["Titulo"], keep="first", inplace=True)
print(df_duplicatas.shape)

(64, 4)
(31, 4)


In [58]:
# realizar mesmo protocolo acima para o df completo, ou seja, ordenar pela coluna "Fonte" para garantir a ordem de preferência Pubmed > Scopus > Wiley > IEEE e depois remover duplicatas mantendo a primeira ocorrência
df_completo["Fonte"] = pd.Categorical(df_completo["Fonte"], categories=["PubMed", "Scopus", "Wiley", "IEEE"], ordered=True)
df_completo = df_completo.sort_values("Fonte")
print(df_completo.shape)
df_completo.drop_duplicates(subset=["Titulo"], keep="first", inplace=True)
print(df_completo.shape)

(562, 4)
(529, 4)


In [59]:
# df completo para csv após remoção de duplicatas
caminho = r"C:/Users/Hychiro/Documents/Mestrado/Mestrado/Mestrado/Review/SLR-BCI-temporal-filters/structured_data/dataframe_completo_sem_duplicatas.csv"
df_completo.to_csv(caminho, index=False, encoding="utf-8")
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 529 entries, 437 to 280
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Titulo             529 non-null    object  
 1   Autores            528 non-null    object  
 2   Ano de Publicacao  529 non-null    int64   
 3   Fonte              529 non-null    category
dtypes: category(1), int64(1), object(2)
memory usage: 17.2+ KB
